# S6E7 | FT-Transformer v2 — Predicting Student Health Risk 🩺

**Playground Series S6E7** asks us to classify each student's **`health_condition`** as
**at-risk**, **unhealthy**, or **fit** from 13 health-behaviour features. The metric is
**Balanced Accuracy** — the *mean of the per-class recall* — under wild class imbalance
(at-risk **~86%**, unhealthy **~8%**, fit **~6%**).

Most public notebooks here are gradient-boosted trees. This one is a **neural network**:
**FT-Transformer v2** — a Feature Tokenizer + Transformer (Gorishniy et al. 2021) — every feature becomes a token, a [CLS] token attends over all of them — trained here on the raw 13 columns **plus 39 per-value target-encoding features**, as a 4-member in-model ensemble.

We use **[`masamlp`](https://pypi.org/project/masamlp/)**, a small LightGBM-style PyTorch tabular
library: `MasaClassifier(model="ft_transformer")` takes the raw DataFrame — categoricals become
learned **embeddings** (missing = its own index), numeric NaN is median-imputed — and trains on
**GPU or CPU with no code change** (`device="auto"`). We show the two levers our paired CV
actually confirmed for NNs on this data (**PLR numeric embeddings** and **explicit missingness
flags**), the **prior-correction decision rule** (the biggest lever on this competition), 7-fold
CV, ensemble-ready outputs, and an honest, evidence-backed list of **what did *not* help**.

> This notebook (and the `masamlp` library it uses) were developed with heavy use of
> **[Claude Code](https://claude.com/claude-code)**.

## Credit where credit is due 🙏

This notebook exists because of the public **"FT-Transformer + PLR embeddings — Single Model"**
notebook on this competition. Go read it: it is a clean, from-scratch FT-Transformer with periodic
numerical embeddings, and it is what sent us back to a feature we had wrongly written off.

We reproduced it under our paired-CV protocol — identical folds, full 690,088 rows, no
best-epoch-on-validation selection — and then took it apart one piece at a time. The honest result
is not the one its title suggests:

| its ingredient | measured, paired, at full scale |
|---|---|
| **PLR numerical embeddings** (its headline) | **null** (+0.00003, 3 of 5 folds) vs the shared-linear `plr-lite` we already shipped |
| **per-value target encoding** (barely mentioned) | **+0.0012, 5 of 5 folds** — the entire difference |
| class-weighted loss | an exact substitute for the post-hoc `beta` rule (ΔBA +0.00001) |
| early stopping on validation balanced accuracy | **+0.0003 of optimism**, not of skill |
| the interaction of PLR × target encoding | +0.00002 — none |

None of that diminishes the notebook; it found a real lever and shared it. It does mean the lever is
not the one it names — and that our own lab had the same feature filed as a dead end. Sections 6b
and 11 tell that story, because a result you can't reproduce the *reason* for is a result you don't
own yet.

## What is an FT-Transformer — and what does v2 change? 🧠

The **FT-Transformer** (Gorishniy et al., *Revisiting Deep Learning Models for Tabular Data*, 2021)
does to a table what BERT does to a sentence. Each feature — numeric or categorical — is mapped to
its own `d_block`-dimensional **token**; a learned **[CLS]** token is prepended; a stack of PreNorm
transformer blocks lets every feature attend to every other; the [CLS] token is read out through a
linear head. Interactions are *learned by attention* rather than carved out by splits.

**v2 changes three things, and only one of them matters.**

1. **`te_mode="direct"` — 39 per-value target-encoding features (§6b).** This is the lever:
   **+0.0012 balanced accuracy, on all 5 folds**, at full scale.
2. **A different training budget**: 16 fixed epochs at batch 4096 with a cosine schedule, instead of
   64 epochs at batch 1024 with early stopping. It is worth **+0.0005** and runs **11× faster** —
   2,160 optimizer steps instead of ~20,000. Big-batch, short-schedule, no early stopping.
3. **`n_ens=4` — an in-model ensemble.** Four independently-initialised transformers train inside
   one estimator and average their probabilities. Judge it honestly and it *fails* on balanced
   accuracy (+0.00017, only 4 of 7 folds — under BA's own ±0.0002 discreteness) while *passing*
   decisively on balanced logloss (7 of 7 folds, on two different fold partitions). We ship it for
   the probabilities, not for the leaderboard, and we say so.

What v2 does **not** change: the PLR numerical embeddings stay `plr-lite`. The full per-feature PLR
of the source notebook measured +0.00003 (3/5) — indistinguishable from the cheaper variant.

## 1. Setup & environment

`masamlp` is pure PyTorch (already on Kaggle) + scikit-learn. Enable **Internet** for `pip`, and
prefer a **GPU accelerator** (the notebook auto-detects it; CPU works too, just slower —
~15 min per fold on the Kaggle T4 with `N_ENS = 4` (four nets are trained per fold). Set `n_ens=1` in `MODEL_PARAMS` for a ~4x faster run: it costs nothing measurable in balanced accuracy, only in probability quality).

In [ ]:
!pip install -q masamlp==0.3.0
!pip install -q catstat==0.4.0

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, classification_report

import masamlp
from masamlp import MasaClassifier

print("masamlp", masamlp.__version__, "| torch", torch.__version__,
      "| device:", "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
import json
from pathlib import Path


## 2. Configuration

Two findings from our local paired cross-validation shaped this config. **(1)** Two NN-specific
levers cleared the adoption bar (+0.002 balanced accuracy, sign-consistent across folds):
**`num_embedding="plr-lite"`** — periodic-linear numeric embeddings (each scalar becomes a small
learned vector, so the net models non-linear numeric effects *before* the first layer) — and
**explicit missingness flags** (section 6). Target encoding, class weighting, ordinal integer
encoding and one-hot cats all failed the same bar (section 11). Reference geometry at `d_block=128`, 2 blocks, 8 heads (the source notebook's, smaller than the library default 192/3 and no worse). `early_stopping_rounds=None` is deliberate: a fixed 16-epoch budget, so nothing about the held-out fold ever picks a checkpoint.
**(2)** As on every model on this competition, the decisive post-processing lever is
**prior-correcting the decision rule** (section 8), so we search a `beta` grid bracketing `1.0`.

In [ ]:
SEED = 4242
N_SPLITS = 7                       # 7-fold StratifiedKFold(rs=42): stabilises minority-class recall
INPUT_DIR = "/kaggle/input/playground-series-s6e7"   # competition data (use "Add Data" if missing)
TARGET = "health_condition"
ID_COL = "id"

# Prior-correction grid for balanced accuracy (section 8). Tuned on OOF; extends to 2.5 because
# the extreme majority prior (~0.86) pushes the held-out optimum HIGH.
BETA_GRID = [0.0]  # no post-hoc threshold tuning; raw argmax only

# masamlp estimator config (== the screened config in ftt/model.py)
MODEL_PARAMS = dict(
    model='ft_transformer',
    n_epochs=16,
    batch_size=4096,
    learning_rate=0.001,
    weight_decay=1e-05,
    optimizer='adamw',
    optimizer_betas=None,
    lr_scheduler='cosine',
    weight_decay_schedule='none',
    grad_clip=None,
    num_embedding='plr-lite',
    numeric_scaler='quantile',
    cat_encoding='embedding',
    class_weight='balanced',
    label_smoothing=0.0,
    early_stopping_rounds=None,
    eval_metric='multi_logloss',
    n_ens=4,
    device='auto',
    amp='auto',
    verbose=0,
    ens_mode='loop',
    eval_batch_size=2048,
    model_params={'d_block': 128, 'n_blocks': 2, 'attention_n_heads': 8, 'n_frequencies': 24, 'sigma': 0.1},
)
EXPERIMENT_ID = "ftt_balanced_seed4242_7fold"
MODEL_ROOT = Path('model_checkpoints')
MODEL_ROOT.mkdir(exist_ok=True)


## 3. The model — one estimator, raw frame in

`masamlp.MasaClassifier` behaves like LightGBM's sklearn estimator: `fit(X, y, eval_set=...)`
with early stopping, `predict_proba`, string labels handled internally
(`classes_ == np.sort(unique(y))`, so probability columns line up with `classes`). Internally it:

- **embeds the 6 categorical columns** (per-column `nn.Embedding`; missing/unseen = a reserved
  index — no target encoding, no one-hot needed);
- **quantile-scales the numerics** and **median-imputes** their NaN (fit on the training fold
  only — which is exactly why the explicit missingness flags in section 6 help: without them the
  net cannot see *which* values were imputed);
- applies **`plr-lite` numeric embeddings** — the screened lever from section 2;
- trains with AdamW + early stopping on the held-out fold's `multi_logloss` (probability quality
  is what the `beta` rule in section 8 exploits — stopping on balanced accuracy directly was far
  worse in our CV).

In [ ]:
def make_model(seed: int = SEED) -> MasaClassifier:
    return MasaClassifier(**MODEL_PARAMS, categorical_features=CAT_COLS, random_state=seed)

## 4. Load the competition data

In [ ]:
# Robust to either mount path (/kaggle/input/playground-series-s6e7 or .../competitions/...).
if not os.path.exists(f"{INPUT_DIR}/train.csv"):
    hits = glob.glob("/kaggle/input/**/train.csv", recursive=True)
    if hits:
        INPUT_DIR = os.path.dirname(hits[0])

train = pd.read_csv(f"{INPUT_DIR}/train.csv")
test = pd.read_csv(f"{INPUT_DIR}/test.csv")
sample_submission = pd.read_csv(f"{INPUT_DIR}/sample_submission.csv")

# Schema-adaptive fallbacks if the official column names ever differ.
if TARGET not in train.columns:
    TARGET = train.columns[-1]
if ID_COL not in train.columns:
    ID_COL = train.columns[0]

print("train:", train.shape, "| test:", test.shape)
print("target:", TARGET, "| id:", ID_COL)
train.head()

## 5. A look at the target — why *balanced* accuracy? ⚖️

Balanced accuracy is the **average of the per-class recall**. The classes here are extremely
imbalanced (at-risk ~86%, unhealthy ~8%, fit ~6%), so a model that always predicts the majority
class scores ~0.86 on plain accuracy but only **1/3 ≈ 0.333** on balanced accuracy. To win we
must recall **unhealthy** and **fit** — the two tiny classes — as well as at-risk. Section 8 is
precisely the knob that buys that.

In [ ]:
dist = train[TARGET].value_counts(normalize=True).round(4)
print("Class distribution:")
print(dist.to_string())
print(f"\nAlways-predict-majority balanced accuracy = {1 / train[TARGET].nunique():.4f}")
print(f"Plain accuracy of that same constant guess = {dist.max():.4f}  (why we DON'T use accuracy)")
train.describe(include="all").T

## 6. Features — the raw 13, and nothing bolted on 🧩

Every S6E7 column carries injected missing values (1–12% per column). v1 handed the network an
explicit `n_missing` count and 13 `na_<col>` flags, because a neural net **median-imputes** numeric
NaN and so erases *which* entries were missing — worth **+0.0025** there.

v2 drops them. The target encoder in the next section treats `NaN` as its own value and encodes it
per column, which is a strictly richer statement of the same fact. Beyond that we still do **not**
hand-engineer: no arithmetic feature soup, no ordinal recoding of the ordered categories — each of
those failed the same paired CV (section 11).

In [ ]:
features = [c for c in train.columns if c not in (ID_COL, TARGET)]
CAT_COLS = [c for c in features if train[c].dtype == object]
NUM_COLS = [c for c in features if c not in CAT_COLS]
X, y = train[features], train[TARGET]
X_test = test[features]
classes = np.sort(y.unique())
prior = y.value_counts().reindex(classes).to_numpy() / len(y)
print(f"{len(features)} raw features | embedded categoricals ({len(CAT_COLS)}): {CAT_COLS}")
print("classes:", list(classes), "| prior:", prior.round(4).tolist())

### 6b. Per-value target encoding — the lever this notebook is built around 🎯

Every one of the 13 columns — the 6 categoricals *and* the 7 numerics — gets replaced, for each
class, by the mean of that class's indicator over the rows sharing its **exact value**. Thirteen
columns × three classes = **39 new numeric features**.

Encoding a *numeric* per exact value looks wrong: `step_count` has 12,807 distinct values. It works
here because S6E7's generator **resampled real discrete values** from the source dataset rather than
adding noise — so each numeric is secretly a high-cardinality categorical, and at 690k rows there
are enough rows per value for the mean to carry signal.

That "at 690k rows" is the whole story. Screened at 70k this feature is **noise** (−0.0017), and our
own lab wrote it off on that evidence. Re-measured at full scale, paired, it is worth **+0.0012 on
this architecture, on all 5 folds** — and it is what separates this notebook from v1. Two lessons
we paid for: *a per-value encoder cannot be screened at small n*, and *a lever tested only inside a
composition has not been tested* (ours was bundled with a feature that cost −0.0069, and the two
cancelled to a convincing zero).

`catstat`'s `TargetEncoder` fits **inside each training fold** (out-of-fold within that fold, so a
row never encodes itself) and only *transforms* the validation and test rows. Nothing about the
held-out fold reaches the encoder.

In [ ]:
from catstat import TargetEncoder

TE_COLS = CAT_COLS + NUM_COLS          # all 13 raw columns, encoded by their exact value


def fit_te(Xtr, ytr, Xva, Xte):
    """Fit the encoder on the TRAINING fold only; transform val/test. Returns three frames."""
    codes = pd.Series(ytr).map({c: i for i, c in enumerate(classes)}).to_numpy()
    enc = TargetEncoder(random_state=SEED, cols=TE_COLS, stats=("mean",),
                        target_type="multiclass", smooth="auto", numeric="direct")
    parts = [np.asarray(enc.fit_transform(Xtr[TE_COLS], codes), dtype="float32")]
    parts += [np.asarray(enc.transform(F[TE_COLS]), dtype="float32") for F in (Xva, Xte)]
    names = [f"te_{i}" for i in range(parts[0].shape[1])]
    out = []
    for F, A in zip((Xtr, Xva, Xte), parts):
        out.append(pd.concat([F.reset_index(drop=True),
                              pd.DataFrame(A, columns=names)], axis=1))
    return out


print(f"target encoding {len(TE_COLS)} columns x {len(classes)} classes "
      f"-> {len(TE_COLS) * len(classes)} extra numeric features (fit per fold)")

## 7. 7-fold cross-validation with FT-Transformer v2

A fresh model per fold — every learned statistic (embedding tables, numeric scalers, imputation
medians, target-encoding tables) comes from **that fold's training data only**, so there is **no leakage**. The net trains a **fixed** number of epochs — no early stopping, so no statistic of the held-out fold ever selects a checkpoint. (Restoring the best-validation epoch and then scoring on that same fold is worth ~**+0.0003** of pure optimism on this dataset; we decline it.)
We store the **out-of-fold (OOF) probabilities** for calibration and stacking. The CV loop is a
small reusable function (the seed-bagging bonus reuses it). Runtime: ~15 min per fold on the Kaggle T4 with `N_ENS = 4` (four nets are trained per fold). Set `n_ens=1` in `MODEL_PARAMS` for a ~4x faster run: it costs nothing measurable in balanced accuracy, only in probability quality.

In [ ]:
def cross_validate(X, y, X_test, seed=SEED):
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    oof = np.zeros((len(X), len(classes)))
    test_proba = np.zeros((len(X_test), len(classes)))
    fold_scores = []
    for fold, (tr_idx, va) in enumerate(skf.split(X, y)):
        Xtr, Xva, Xte = fit_te(X.iloc[tr_idx], y.iloc[tr_idx], X.iloc[va], X_test)
        model = make_model(seed)
        model.fit(Xtr, y.iloc[tr_idx], eval_set=[(Xva, y.iloc[va])])
        model.save_model(str(MODEL_ROOT / f'fold_{fold}'))
        assert list(model.classes_) == list(classes)   # proba columns align with `classes`
        oof[va] = model.predict_proba(Xva)
        test_proba += model.predict_proba(Xte) / N_SPLITS
        fold_ba = balanced_accuracy_score(y.iloc[va], classes[oof[va].argmax(1)])
        print(f"  fold {fold}: raw balanced accuracy = {fold_ba:.5f}")
        fold_scores.append(float(fold_ba))
    return oof, test_proba, model, fold_scores


oof_proba, test_proba, model, fold_scores = cross_validate(X, y, X_test)
raw_ba = balanced_accuracy_score(y, classes[oof_proba.argmax(1)])
print(f"\nOOF balanced accuracy (raw argmax) = {raw_ba:.5f}")

## 8. Calibrate the decision rule — the biggest lever by far 🎯

`argmax(proba)` maximizes plain accuracy and over-predicts the majority class, which *wrecks*
balanced accuracy here. Dividing each probability by the **class prior** raised to a power
`beta` — i.e. predict

$$\hat{y} = \arg\max_k \; \frac{p_k}{\text{prior}_k^{\,\beta}}$$

shifts the decision toward the rarer classes. `beta = 0` is the plain argmax; `beta = 1` predicts
under a **uniform** prior. We pick `beta` on the **out-of-fold** predictions (no refit) and apply
the same value to the test set. A neural softmax is calibrated differently from a tree's, so we
re-tune `beta` for *this* model; on the full data the optimum sits near
**beta ≈ 1.05**. The payoff is enormous — raw argmax
≈ **0.8919** becomes ≈ **0.95067** after the rule
(**+0.059**), dwarfing every other lever on this competition.

In [ ]:
def tune_beta(oof, label=""):
    scores = {b: balanced_accuracy_score(y, classes[(oof / prior ** b).argmax(1)]) for b in BETA_GRID}
    best = max(scores, key=scores.get)
    for b, s in scores.items():
        print(f"  beta={b:>4}: OOF balanced accuracy = {s:.5f}" + ("  <-- best" if b == best else ""))
    print(f"\n{label}lift: {scores[0.0]:.5f} (raw argmax) -> {scores[best]:.5f} "
          f"(+{scores[best] - scores[0.0]:.5f}) at beta={best}")
    return best


best_beta = tune_beta(oof_proba)
oof_pred = classes[(oof_proba / prior ** best_beta).argmax(1)]
print(classification_report(y, oof_pred, digits=4))

## 9. Build the submission + ensemble-ready outputs

Alongside `submission.csv` (the calibrated labels) we save the **raw probabilities** so this
neural member can be **blended with the tree models later** — the whole reason to bring an NN to
a tree-dominated competition. `oof_ftt.csv` holds the out-of-fold *train* probabilities (the
leakage-free counterpart of the test predictions) and `testpred_ftt.csv` the fold-averaged *test*
probabilities, each with an `id` column and one column per class. A meta-learner fit on the
stacked OOF then generalises to the test.

In [ ]:
pred_labels = classes[(test_proba / prior ** best_beta).argmax(1)]

sub_target = [c for c in sample_submission.columns if c.lower() != ID_COL.lower()][-1]
submission = sample_submission.copy()
submission[sub_target] = pred_labels
submission.to_csv("submission.csv", index=False)

# Ensemble-ready raw probabilities (one column per class) for blending across model families.
oof_df = pd.DataFrame(oof_proba, columns=list(classes))
oof_df.insert(0, ID_COL, train[ID_COL].values)
oof_df.to_csv("oof_ftt.csv", index=False)

testpred_df = pd.DataFrame(test_proba, columns=list(classes))
testpred_df.insert(0, ID_COL, test[ID_COL].values)
testpred_df.to_csv("testpred_ftt.csv", index=False)

print(f"applied beta={best_beta}; predicted label counts:")
print(submission[sub_target].value_counts().to_string())
print("wrote: submission.csv, oof_ftt.csv, testpred_ftt.csv")
summary = {
    'experiment_id': EXPERIMENT_ID,
    'uses_only_competition_data': True,
    'seed': SEED,
    'folds': N_SPLITS,
    'class_weight': 'balanced',
    'decision_rule': 'raw_argmax',
    'raw_oof_balanced_accuracy': float(raw_ba),
    'fold_scores': fold_scores,
}
Path('training_summary.json').write_text(json.dumps(summary, indent=2))
submission.head()

## 10. (Optional bonus) Seed-bagging — free variance reduction 🎲

A neural net carries **variance** from its random seed (weight init, batch order, the fold
shuffle). Averaging the **out-of-fold probabilities** of the *same* configuration under a few
seeds cancels some of that noise — a textbook **bagging** lift that needs no new modelling, only
extra compute. The cell **reuses the main run** as the first member, trains only the *extra*
seeds, and keeps the bagged result only if it **matches or beats** the single-model OOF, so
enabling it is safe.

In [ ]:
SEED_BAG_EXTRA = []   # extra seeds beyond the main SEED (empty = skip)

n_members = 1 + len(SEED_BAG_EXTRA)
oof_bag, test_bag = oof_proba.copy(), test_proba.copy()
for s in SEED_BAG_EXTRA:
    o, t, _, _ = cross_validate(X, y, X_test, seed=s)
    oof_bag += o
    test_bag += t
oof_bag /= n_members
test_bag /= n_members

if SEED_BAG_EXTRA:
    beta_bag = tune_beta(oof_bag, label="[seed-bag] ")
    base_ba = balanced_accuracy_score(y, classes[(oof_proba / prior ** best_beta).argmax(1)])
    bag_ba = balanced_accuracy_score(y, classes[(oof_bag / prior ** beta_bag).argmax(1)])
    print(f"\nbalanced accuracy  single={base_ba:.5f}  ->  {n_members}-seed bag={bag_ba:.5f}  "
          f"(delta={bag_ba - base_ba:+.5f})")

    # Adopt the bagged outputs only if they matched or beat the single-model OOF.
    if bag_ba >= base_ba:
        submission[sub_target] = classes[(test_bag / prior ** beta_bag).argmax(1)]
        submission.to_csv("submission.csv", index=False)
        print(f"seed-bagged submission written (beta={beta_bag}; matched or beat the single model).")
else:
    print("seed-bag disabled (the in-model ensemble (n_ens=4) already averages 4 independently-seeded nets).")

## 11. What did *not* help (tested under the same paired CV) 🔬

Honesty saves you time — these are all **dead ends on this dataset**, each measured under
identical folds against the same baseline, so you can skip them:

- **Target encoding was the one "dead end" that wasn't.** Our lab measured it at 70k rows
  (−0.0017) and again at full scale — but only *composed* with numeric-as-categorical embeddings, a
  feature that costs −0.0069. The two cancelled to a convincing zero and we filed it as null. Alone,
  at 690k rows, it is **+0.0012 on 5 of 5 folds** and it is section 6b of this notebook. Two rules
  came out of that: **a per-value encoder cannot be screened at small n**, and **a lever tested only
  inside a composition has not been tested.**
- **Full per-feature PLR embeddings** (`num_embedding="plr"`, the source notebook's headline):
  **+0.00003, 3/5 folds** against the shared-linear `plr-lite` this notebook ships. Their
  interaction with target encoding is +0.00002. Neither is a lever.
- **Early-stopping on balanced accuracy directly** was far worse (−0.012): the metric is a noisy,
  discrete stop signal. Stop on `multi_logloss` (probability quality) and let the `beta` rule
  convert it into balanced accuracy.
- **Training-time class weighting** (`class_weight="balanced"`) is an *exact* substitute for the
  post-hoc `beta` rule at this scale: ΔBA **+0.00001**, and `beta*` simply moves from 1.05 to 0.05.
  We leave it off — not because it hurts accuracy, but because a class-weighted net emits the
  *balanced* posterior `p(y|x)/prior(y)` rather than `p(y|x)`, and that displaced prior quietly
  corrupts anything downstream that scores probabilities (a stacker, or balanced logloss itself).
- **Ordinal integer re-encoding** of the 3 naturally-ordered categories (stress, sleep quality,
  activity: low<medium<high) — the embeddings evidently recover what order carries (−0.0006).
- **One-hot categoricals** instead of embeddings: −0.001. **Lower/higher learning rate** (3e-4 /
  3e-3), **cosine schedule**, **longer budget** (128 epochs): all within noise of the base —
  early stopping at 64 epochs is already binding.
- **The original 50k source dataset as extra rows** — S6E7 was synthesised from a real dataset,
  but the competition already gives ~14× the data with the same class mix (a confirmed null for
  every model family here).
- **`n_ens=8`** — out of memory on a T4 (vmapping 8 attention stacks), and `ens_mode="vectorized"`
  is unusable for attention architectures on CUDA at all (`torch.vmap` over flash-attention raises
  *"LSE is not correctly aligned"*). `ens_mode="loop"` is the same estimator at 4× the wall clock.
- **Explicit missingness flags** (`n_missing` + 13 `na_*` columns) — the lever v1 ships. With
  per-value target encoding they are redundant: the encoder gives `NaN` its own level and encodes it.
- **Best-epoch-on-validation restore** — worth **+0.0003** of balanced accuracy that evaporates the
  moment you score on a fold the checkpoint didn't choose itself. Fixed epochs; no peeking.

## 12. Where to go next (push your CV) 🚀

Ranked by the CV deltas we measured (paired folds, balanced accuracy):

- **Prior-correction `beta`** (section 8) — the dominant lever by a wide margin, already built in.
- **Seed-bagging** (section 10) — a small free bonus.
- **Ensemble across model families — this is why a neural member is here.** This notebook writes
  `oof_ftt.csv` / `testpred_ftt.csv` precisely so you can stack it with the tree models. Two
  gradient-boosted trees agree almost perfectly (OOF correlation ≈ 0.999); this FT-Transformer v2's OOF
  correlates ≈ **0.9985** with an XGBoost of the same CV (they disagree on
  **0.64%** of rows) — genuine independence a second tree seed cannot
  provide, raw material for a stacking meta-learner.

The model auto-selects **GPU or CPU** (`device="auto"`); on Kaggle pick the GPU accelerator for
speed.

In [ ]:
# Persist and reload the fitted estimator with masamlp's native save/load (JSON manifest +
# tensors, no pickle) and confirm the round-trip.
model.save_model("masamlp_ftt_model")
reloaded = MasaClassifier.load_model("masamlp_ftt_model")
_, _, X_demo = fit_te(X, y, X.iloc[:5], X_test.iloc[:5])
print("reloaded matches:",
      bool((reloaded.predict(X_demo) == model.predict(X_demo)).all()))